# Bước 6: Huấn luyện mô hình LightGBM Classifier

Notebook này thực hiện huấn luyện mô hình **LightGBM Classifier** dựa trên tập dữ liệu đã được tiền xử lý.

## 1. Import thư viện

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

print("Imported libraries successfully.")

## 2. Đọc dữ liệu và chuẩn bị đặc trưng

In [ ]:
data_path = 'data/processed/Student_Performance_processed.csv'
for _ in range(5):
    if os.path.exists(data_path):
        break
    data_path = os.path.join('..', data_path)

df_encoded = pd.read_csv(data_path)

objective_features = [
    'age', 'study_hours', 'attendance_percentage',
    'math_score', 'science_score', 'english_score',
    'school_type', 'internet_access', 'extra_activities',
    'parent_education',
    'gender_male', 'gender_female', 'gender_other',
    'travel_time_<15 min', 'travel_time_15-30 min', 'travel_time_30-60 min', 'travel_time_>60 min',
    'study_method_notes', 'study_method_textbook', 'study_method_group study',
    'study_method_coaching', 'study_method_mixed', 'study_method_online videos'
]

df_encoded['study_hours_x_attendance'] = df_encoded['study_hours'] * df_encoded['attendance_percentage']
df_encoded['study_hours_squared'] = df_encoded['study_hours'] ** 2
df_encoded['attendance_squared'] = df_encoded['attendance_percentage'] ** 2

subject_score_features = ['math_score', 'science_score', 'english_score']
df_encoded['subject_score_mean'] = df_encoded[subject_score_features].mean(axis=1)
df_encoded['subject_score_min'] = df_encoded[subject_score_features].min(axis=1)
df_encoded['subject_score_max'] = df_encoded[subject_score_features].max(axis=1)
df_encoded['subject_score_range'] = df_encoded['subject_score_max'] - df_encoded['subject_score_min']
df_encoded['subject_score_std'] = df_encoded[subject_score_features].std(axis=1)

objective_features += [
    'study_hours_x_attendance',
    'study_hours_squared',
    'attendance_squared',
    'subject_score_mean',
    'subject_score_min',
    'subject_score_max',
    'subject_score_range',
    'subject_score_std'
]

X = df_encoded[objective_features]
y = df_encoded['final_grade']

print(f"Loaded processed data: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns.")
print(f"Feature count: {len(objective_features)}")

## 3. Chia tập dữ liệu

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train set: {X_train.shape[0]} rows, Test set: {X_test.shape[0]} rows")

## 4. Huấn luyện mô hình

In [ ]:
model = LGBMClassifier(
    objective='multiclass',
    n_estimators=180,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=7,
    min_child_samples=50,
    subsample=0.9,
    subsample_freq=1,
    colsample_bytree=0.85,
    class_weight='balanced',
    reg_alpha=0.2,
    reg_lambda=5.0,
    random_state=42,
    n_jobs=1,
    verbose=-1
)
model.fit(X_train, y_train)
print("Training completed.")

## 5. Đánh giá mô hình

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
target_labels = sorted(y.unique())

print(f"Accuracy: {acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_labels))

y_train_pred = model.predict(X_train)
f1_train = f1_score(y_train, y_train_pred, average='macro')
f1_test = f1_score(y_test, y_pred, average='macro')
print("\nOverfitting Check")
print(f"Macro-F1 Train : {f1_train:.4f}")
print(f"Macro-F1 Test  : {f1_test:.4f}")
print(f"Difference     : {f1_train - f1_test:.4f}")

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_f1_scores = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
print(f"\nMean CV F1 Score: {cv_f1_scores.mean():.4f} (+/- {cv_f1_scores.std():.4f})")

## 6. Trực quan hóa kết quả

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=target_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=target_labels, yticklabels=target_labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - LightGBM Classifier')
plt.tight_layout()
plt.show()

## 7. Lưu mô hình

In [ ]:
models_dir = 'models'
for _ in range(5):
    if os.path.exists(models_dir):
        break
    models_dir = os.path.join('..', models_dir)

os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, 'lightgbm_model.pkl')
joblib.dump(model, model_path)
print(f"LightGBM model saved successfully to: {model_path}")